In [ ]:
%pip install pandas seaborn matplotlib

In [ ]:
import pandas as pd
import os, re, sys
import numpy as np

## List-to-method correspondences

In [ ]:
def get_method_dict(file):
    method_dict = {'average' : 'average'}
    with open(file, 'r') as f:
        lines = [line.rstrip() for line in f.readlines()]
        i = 1
        for l in lines:
            match = re.search(r'^.+attrib=([^_]+)_mpd.+_thresh=([^_]+)_.+$', l)
            method = match.group(1)
            threshold = match.group(2)
            key = 'list' + str(i)
            method_dict[key] = method + '_' + threshold
            i += 1
    return method_dict


In [ ]:
bcms_setimes_methods = get_method_dict('randomized_lists/randomized_lists.bcms_setimes.txt')
bcms_twitter_methods = get_method_dict('randomized_lists/randomized_lists.bcms_twitter.txt')
estonian_voro_methods = get_method_dict('randomized_lists/randomized_lists.estonian_voro.txt')
finnish_suomi24_methods = get_method_dict('randomized_lists/randomized_lists.finnish.txt')
german_jodel_methods = get_method_dict('randomized_lists/randomized_lists.jodel.txt')
greek_dialects_methods = get_method_dict('randomized_lists/randomized_lists.greek.txt')
scandinavian_slide_methods = get_method_dict('randomized_lists/randomized_lists.scandinavian.txt')

In [ ]:
bcms_setimes_methods_df = pd.DataFrame.from_dict(bcms_setimes_methods, orient='index', columns=['method'])
bcms_setimes_methods_df

In [ ]:
bcms_twitter_methods_df = pd.DataFrame.from_dict(bcms_twitter_methods, orient='index', columns=['method'])
bcms_twitter_methods_df

In [ ]:
estonian_voro_methods_df = pd.DataFrame.from_dict(estonian_voro_methods, orient='index', columns=['method'])
estonian_voro_methods_df

In [ ]:
finnish_suomi24_methods_df = pd.DataFrame.from_dict(finnish_suomi24_methods, orient='index', columns=['method'])
finnish_suomi24_methods_df

In [ ]:
greek_dialects_methods_df = pd.DataFrame.from_dict(greek_dialects_methods, orient='index', columns=['method'])
greek_dialects_methods_df

In [ ]:
german_jodel_methods_df = pd.DataFrame.from_dict(german_jodel_methods, orient='index', columns=['method'])
german_jodel_methods_df

In [ ]:
scandinavian_slide_methods_df = pd.DataFrame.from_dict(scandinavian_slide_methods, orient='index', columns=['method'])
scandinavian_slide_methods_df

## Importing all lists and labelling them with list ID

In [ ]:
def make_dataset_df(folder):
    dfs = list()
    files = os.listdir(folder)
    for file in files:
        print(file)
        if file.endswith('.csv'):
            id = re.sub('.csv', '', file)
            df = pd.read_csv(folder + '/' + file)
            df['file'] = id
            dfs.append(df)
        elif file.endswith('.txt'):
            id = re.sub('.txt', '', file)
            df = pd.read_csv(folder + '/' + file, sep='\t')
            df['file'] = id
            dfs.append(df)
    full_df = pd.concat(dfs, ignore_index=True)
    return full_df

In [ ]:
def get_number_annots(df):
    number_annots = df.value_counts('file').to_frame()
    number_annots = number_annots.rename(columns={number_annots.columns[0]: "total"})
    return number_annots

In [ ]:
bcms_twitter_df = make_dataset_df('bcms_twitter')
bcms_twitter_df

In [ ]:
bcms_setimes_df = make_dataset_df('bcms_setimes')
bcms_setimes_df

In [ ]:
estonian_voro_df = make_dataset_df('eesti-voro')
estonian_voro_df

In [ ]:
german_jodel_df = make_dataset_df('german_jodel')
german_jodel_df

In [ ]:
german_jodel_df['other_variety'].value_counts()

In [ ]:
greek_dialects_df = make_dataset_df('greek_dialects')
greek_dialects_df

In [ ]:
finnish_suomi24_df = make_dataset_df('finnish_suomi24')
finnish_suomi24_df

In [ ]:
scandinavian_slide_df = make_dataset_df('scandinavian_slide')
scandinavian_slide_df

# Data preparation

Aim: data frame containing the percentage of lexical shibboleths, other shibboleths, non shibboleths and if applicable other annotations

## Preparing bcms_setimes dataframe

In [ ]:
number_annots = get_number_annots(bcms_setimes_df)
number_annots

In [ ]:
bcms_setimes_shib_distro = bcms_setimes_df[(bcms_setimes_df['shibboleth'] == 'yes') & (bcms_setimes_df['feature'] == 'lex')].groupby('file').size().to_frame(name='lex shib')
bcms_setimes_shib_distro['nonlex shib'] = bcms_setimes_df[(bcms_setimes_df['shibboleth'] == 'yes') & (bcms_setimes_df['feature'] != 'lex')].groupby('file').size()
bcms_setimes_shib_distro['non shib'] = bcms_setimes_df[bcms_setimes_df['shibboleth'] == 'no'].groupby('file').size()
bcms_setimes_shib_distro['rest'] = bcms_setimes_df[(bcms_setimes_df['shibboleth'] != 'no') & (bcms_setimes_df['shibboleth'] != 'yes')].groupby('file').size()
bcms_setimes_shib_distro = pd.concat([bcms_setimes_shib_distro, number_annots], axis=1)
bcms_setimes_shib_distro['shibboleths: lexical'] = round(bcms_setimes_shib_distro['lex shib'] / bcms_setimes_shib_distro['total'] * 100, 1)
bcms_setimes_shib_distro['shibboleths: other'] = round(bcms_setimes_shib_distro['nonlex shib'] / bcms_setimes_shib_distro['total'] * 100, 1)
bcms_setimes_shib_distro['non shibboleths'] = round(bcms_setimes_shib_distro['non shib'] / bcms_setimes_shib_distro['total'] * 100, 1)
bcms_setimes_shib_distro['other'] = round(bcms_setimes_shib_distro['rest'] / bcms_setimes_shib_distro['total'] * 100, 1)
bcms_setimes_shib_distro['perc sum'] = bcms_setimes_shib_distro['shibboleths: lexical'] + bcms_setimes_shib_distro['shibboleths: other'] \
+ bcms_setimes_shib_distro['non shibboleths'] + bcms_setimes_shib_distro['other']
bcms_setimes_shib_distro.loc['average'] = round(bcms_setimes_shib_distro.mean(axis=0), 1)
bcms_setimes_shib_distro

In [ ]:
bcms_setimes_shib_distro = pd.concat([bcms_setimes_shib_distro, bcms_setimes_methods_df], axis=1)


In [ ]:
bcms_setimes_shib_distro

In [ ]:
bcms_setimes_shib_distro = bcms_setimes_shib_distro.sort_values(by=['method']).reset_index(names=['list']).set_index('method')
bcms_setimes_shib_distro

In [ ]:
bcms_setimes_overview = bcms_setimes_shib_distro[['shibboleths: lexical', 'shibboleths: other', 'non shibboleths', 'other']]
bcms_setimes_overview

## Preparing bcms_twitter dataset

In [ ]:
number_annots = get_number_annots(bcms_twitter_df)
number_annots

In [ ]:
bcms_twitter_shib_distro = bcms_twitter_df[(bcms_twitter_df['shibboleth'] == 'yes') & (bcms_twitter_df['feature'] == 'lex')].groupby('file').size().to_frame(name='lex shib')
bcms_twitter_shib_distro['nonlex shib'] = bcms_twitter_df[(bcms_twitter_df['shibboleth'] == 'yes') & (bcms_twitter_df['feature'] != 'lex')].groupby('file').size()
bcms_twitter_shib_distro['non shib'] = bcms_twitter_df[bcms_twitter_df['shibboleth'] == 'no'].groupby('file').size()
bcms_twitter_shib_distro['rest'] = bcms_twitter_df[(bcms_twitter_df['shibboleth'] != 'no') & (bcms_twitter_df['shibboleth'] != 'yes')].groupby('file').size()
bcms_twitter_shib_distro = pd.concat([bcms_twitter_shib_distro, number_annots], axis=1)
bcms_twitter_shib_distro['shibboleths: lexical'] = round(bcms_twitter_shib_distro['lex shib'] / bcms_twitter_shib_distro['total'] * 100, 1)
bcms_twitter_shib_distro['shibboleths: other'] = round(bcms_twitter_shib_distro['nonlex shib'] / bcms_twitter_shib_distro['total'] * 100, 1)
bcms_twitter_shib_distro['non shibboleths'] = round(bcms_twitter_shib_distro['non shib'] / bcms_twitter_shib_distro['total'] * 100, 1)
bcms_twitter_shib_distro['other'] = round(bcms_twitter_shib_distro['rest'] / bcms_twitter_shib_distro['total'] * 100, 1)
bcms_twitter_shib_distro['perc sum'] = bcms_twitter_shib_distro['shibboleths: lexical'] + bcms_twitter_shib_distro['shibboleths: other'] \
+ bcms_twitter_shib_distro['non shibboleths'] + bcms_twitter_shib_distro['other']
bcms_twitter_shib_distro.loc['average'] = round(bcms_twitter_shib_distro.mean(axis=0), 1)
bcms_twitter_shib_distro

In [ ]:
bcms_twitter_shib_distro = pd.concat([bcms_twitter_shib_distro, bcms_twitter_methods_df], axis = 1)
bcms_twitter_shib_distro

In [ ]:
bcms_twitter_shib_distro  = bcms_twitter_shib_distro.sort_values(by='method').reset_index(names=['list']).set_index('method')
bcms_twitter_shib_distro

In [ ]:
bcms_twitter_overview = bcms_twitter_shib_distro[['shibboleths: lexical', 'shibboleths: other', 'non shibboleths', 'other']]
bcms_twitter_overview

## Preparing german_jodel dataframe

In [ ]:
number_annots = get_number_annots(german_jodel_df)
number_annots

In [ ]:
german_jodel_df.value_counts('code')

In [ ]:
# Named entities are not shibboleths
# Only single-variety shibboleths count
german_jodel_df['shibboleth'] = 'no'
german_jodel_df.loc[(german_jodel_df['code'] != 'NE') & (german_jodel_df['code'] != 'unmarked') \
& (german_jodel_df['other_variety'].isna()) & (german_jodel_df['note'].isna()) , 'shibboleth'] = 'yes'
# sanity check:
german_jodel_df[(german_jodel_df['code'] == 'lex') & (german_jodel_df['shibboleth'] == 'no')]

In [ ]:
german_jodel_shib_distro = german_jodel_df[(german_jodel_df['shibboleth'] == 'yes') & (german_jodel_df['code'] == 'lex')].groupby('file').size().to_frame(name='lex shib')
german_jodel_shib_distro['nonlex shib'] = german_jodel_df[(german_jodel_df['shibboleth'] == 'yes') & (german_jodel_df['code'] != 'lex')].groupby('file').size()
german_jodel_shib_distro['non shib'] = german_jodel_df[german_jodel_df['shibboleth'] == 'no'].groupby('file').size()
german_jodel_shib_distro.loc[:,'sum'] = german_jodel_shib_distro.sum(axis=1)
german_jodel_shib_distro = pd.concat([german_jodel_shib_distro, number_annots], axis=1)
german_jodel_shib_distro['shibboleths: lexical'] = round(german_jodel_shib_distro['lex shib'] / german_jodel_shib_distro['total'] * 100, 1)
german_jodel_shib_distro['shibboleths: other'] = round(german_jodel_shib_distro['nonlex shib'] / german_jodel_shib_distro['total'] * 100, 1)
german_jodel_shib_distro['non shibboleths'] = round(german_jodel_shib_distro['non shib'] / german_jodel_shib_distro['total'] * 100, 1)
german_jodel_shib_distro['perc_sum'] = german_jodel_shib_distro['shibboleths: lexical'] + german_jodel_shib_distro['shibboleths: other'] + german_jodel_shib_distro['non shibboleths']
german_jodel_shib_distro.loc['average'] = round(german_jodel_shib_distro.mean(axis=0), 1)

german_jodel_shib_distro

In [ ]:
german_jodel_shib_distro = pd.concat([german_jodel_shib_distro, german_jodel_methods_df], axis=1)
german_jodel_shib_distro

In [ ]:
german_jodel_shib_distro = german_jodel_shib_distro.sort_values(by=['method']).reset_index(names=['list']).set_index('method')
german_jodel_shib_distro

In [ ]:
german_jodel_overview = german_jodel_shib_distro[['shibboleths: lexical', 'shibboleths: other', 'non shibboleths']]
german_jodel_overview

## Preparing greek_dialects dataframe

In [ ]:
greek_dialects_df[greek_dialects_df['file'] ==  'list2']

In [ ]:
greek_dialects_df.value_counts('feature')

In [ ]:
number_annots = get_number_annots(greek_dialects_df)
number_annots

In [ ]:
greek_dialects_shib_distro = greek_dialects_df[greek_dialects_df['feature'] == 'Lex'].groupby('file').size().to_frame(name='lex shib')
greek_dialects_shib_distro['nonlex shib'] = greek_dialects_df[(greek_dialects_df['feature'] != 'NE') & (greek_dialects_df['feature'] != 'Unmarked') & (greek_dialects_df['feature'] != 'Lex')].groupby('file').size()
greek_dialects_shib_distro['non shib'] = greek_dialects_df[(greek_dialects_df['feature'] == 'NE') | (greek_dialects_df['feature'] == 'Unmarked')].groupby('file').size()
greek_dialects_shib_distro.loc[:, 'sum'] = greek_dialects_shib_distro.sum(axis=1)
greek_dialects_shib_distro = pd.concat([greek_dialects_shib_distro, number_annots], axis=1)
greek_dialects_shib_distro['shibboleths: lexical'] = round(greek_dialects_shib_distro['lex shib'] / \
                                                          greek_dialects_shib_distro['total'] * 100, 1)
greek_dialects_shib_distro['shibboleths: other'] = round(greek_dialects_shib_distro['nonlex shib'] / \
                                                          greek_dialects_shib_distro['total'] * 100, 1)
greek_dialects_shib_distro['non shibboleths'] = round(greek_dialects_shib_distro['non shib'] / \
                                                          greek_dialects_shib_distro['total'] * 100, 1)
greek_dialects_shib_distro['perc_sum'] = greek_dialects_shib_distro['shibboleths: lexical'] + greek_dialects_shib_distro['shibboleths: other'] \
+ greek_dialects_shib_distro['non shibboleths']
greek_dialects_shib_distro.loc['average'] = round(greek_dialects_shib_distro.mean(axis=0) , 1)

greek_dialects_shib_distro

In [ ]:
greek_dialects_shib_distro = pd.concat([greek_dialects_shib_distro, greek_dialects_methods_df], axis=1)
greek_dialects_shib_distro

In [ ]:
greek_dialects_shib_distro = greek_dialects_shib_distro.sort_values(by=['method']).reset_index(names=['list']).set_index('method')
greek_dialects_shib_distro

In [ ]:
greek_dialects_overview = greek_dialects_shib_distro[['shibboleths: lexical', 'shibboleths: other', 'non shibboleths']]
greek_dialects_overview

## Preparing finnish_suomi24 dataframe

In [ ]:
finnish_suomi24_df

In [ ]:
finnish_suomi24_df.value_counts('label')

In [ ]:
number_annots = get_number_annots(finnish_suomi24_df)
number_annots

In [ ]:
finnish_suomi24_shib_distro = finnish_suomi24_df[finnish_suomi24_df['lex_other_not'] == 'lex'].groupby('file').size().to_frame(name='lex shib')
finnish_suomi24_shib_distro['nonlex shib'] = finnish_suomi24_df[finnish_suomi24_df['lex_other_not'] == 'other'].groupby('file').size()
finnish_suomi24_shib_distro['non shib'] = finnish_suomi24_df[finnish_suomi24_df['lex_other_not'] ==  'not'].groupby('file').size()
finnish_suomi24_shib_distro.loc[:, 'sum'] = finnish_suomi24_shib_distro.sum(axis=1)
finnish_suomi24_shib_distro = pd.concat([finnish_suomi24_shib_distro, number_annots], axis=1)
finnish_suomi24_shib_distro['shibboleths: lexical'] = round(finnish_suomi24_shib_distro['lex shib'] / 
                                                             finnish_suomi24_shib_distro['total'] * 100, 1)
finnish_suomi24_shib_distro['shibboleths: other'] = round(finnish_suomi24_shib_distro['nonlex shib'] / 
                                                             finnish_suomi24_shib_distro['total'] * 100, 1)
finnish_suomi24_shib_distro['non shibboleths'] = round(finnish_suomi24_shib_distro['non shib'] / 
                                                             finnish_suomi24_shib_distro['total'] * 100, 1)
finnish_suomi24_shib_distro['perc_sum'] = finnish_suomi24_shib_distro[['shibboleths: lexical', 'shibboleths: other', 'non shibboleths']].sum(axis=1)
finnish_suomi24_shib_distro.loc['average'] = round(finnish_suomi24_shib_distro.mean(axis=0), 1)


finnish_suomi24_shib_distro

In [ ]:
finnish_suomi24_shib_distro = pd.concat([finnish_suomi24_shib_distro, finnish_suomi24_methods_df], axis=1)
finnish_suomi24_shib_distro

In [ ]:
finnish_suomi24_shib_distro = finnish_suomi24_shib_distro.sort_values(by=['method']).reset_index(names=['list']).set_index('method')
finnish_suomi24_shib_distro

In [ ]:
finnish_suomi24_overview = finnish_suomi24_shib_distro[['shibboleths: lexical', 'shibboleths: other', 'non shibboleths']]
finnish_suomi24_overview

## Prepare estonian-voro dataframe

In [ ]:
estonian_voro_df

In [ ]:
number_annots = get_number_annots(estonian_voro_df)
number_annots

In [ ]:
estonian_voro_df[estonian_voro_df['Lexicon'].str.contains(r'x|\?', na=False)]

In [ ]:
estonian_voro_df["unk"] = 0
estonian_voro_df.loc[(estonian_voro_df['Unknown'].str.contains('x', na=False) | estonian_voro_df['Other'].str.contains('x', na=False)), "unk"] = 1
estonian_voro_df["noshib"] = 0
estonian_voro_df.loc[((estonian_voro_df["unk"] == 0) & estonian_voro_df['Unmarked'].str.contains('x', na=False)), "noshib"] = 1
estonian_voro_df["lexshib"] = 0
estonian_voro_df.loc[((estonian_voro_df["unk"] == 0) & (estonian_voro_df["noshib"] == 0) & estonian_voro_df['Lexicon'].str.contains('x', na=False)), "lexshib"] = 1
estonian_voro_df["nonlexshib"] = 0
estonian_voro_df.loc[((estonian_voro_df["unk"] == 0) & (estonian_voro_df["noshib"] == 0) & (estonian_voro_df["lexshib"] == 0) & estonian_voro_df['Phon-Morph-infl-Morph-der-Spelling'].str.contains(r'x|s', na=False)), "nonlexshib"] = 1

estonian_voro_df.loc[estonian_voro_df["unk"] + estonian_voro_df["noshib"] + estonian_voro_df["lexshib"] + estonian_voro_df["nonlexshib"] == 0, "unk"] += 1

estonian_voro_df[estonian_voro_df["unk"] >= 1]

In [ ]:
estonian_voro_shib_distro = estonian_voro_df[estonian_voro_df['lexshib']==1].groupby('file').size().to_frame(name="lex shib")
estonian_voro_shib_distro['non lex shib'] = estonian_voro_df[estonian_voro_df['nonlexshib']==1].groupby('file').size()
estonian_voro_shib_distro['non shib'] = estonian_voro_df[estonian_voro_df['noshib']==1].groupby('file').size()
estonian_voro_shib_distro['rest'] = estonian_voro_df[estonian_voro_df['unk']==1].groupby('file').size()
estonian_voro_shib_distro.loc[:, 'sum'] = estonian_voro_shib_distro.sum(axis=1)

estonian_voro_shib_distro = pd.concat([estonian_voro_shib_distro, number_annots], axis=1)
estonian_voro_shib_distro['shibboleths: lexical'] = round(estonian_voro_shib_distro['lex shib'] \
                                                               / estonian_voro_shib_distro['total'] * 100, 1)
estonian_voro_shib_distro['shibboleths: other'] = round(estonian_voro_shib_distro['non lex shib'] \
                                                               / estonian_voro_shib_distro['total'] * 100, 1)
estonian_voro_shib_distro['non shibboleths'] = round(estonian_voro_shib_distro['non shib'] \
                                                               / estonian_voro_shib_distro['total'] * 100, 1)
estonian_voro_shib_distro['other'] = round(estonian_voro_shib_distro['rest'] \
                                                               / estonian_voro_shib_distro['total'] * 100, 1)

estonian_voro_shib_distro.loc[:, 'perc_sum'] = estonian_voro_shib_distro[['shibboleths: lexical', 'shibboleths: other', 'non shibboleths', 'other']].sum(axis=1)
estonian_voro_shib_distro.loc['average'] = round(estonian_voro_shib_distro.mean(axis=0), 1)

estonian_voro_shib_distro

In [ ]:
estonian_voro_overview = estonian_voro_shib_distro[['shibboleths: lexical', 'shibboleths: other', 'non shibboleths', 'other']]
estonian_voro_overview

## Prepare scandinavian_slide dataframe

In [ ]:
scandinavian_slide_df

In [ ]:
scandinavian_slide_df[(scandinavian_slide_df['NE'] == True) & (scandinavian_slide_df['NonLexShib'] == True)]

In [ ]:
number_annots = get_number_annots(scandinavian_slide_df)
number_annots

In [ ]:
scandinavian_slide_shib_distro = scandinavian_slide_df[scandinavian_slide_df['LexShib'] == True].groupby('file').size().to_frame(name='lex shib')
scandinavian_slide_shib_distro['nonlex shib'] = scandinavian_slide_df[scandinavian_slide_df['NonLexShib'] == True].groupby('file').size()
scandinavian_slide_shib_distro['non shib'] = scandinavian_slide_df[(scandinavian_slide_df['NotShib'] ==  True)].groupby('file').size()
scandinavian_slide_shib_distro['rest'] = scandinavian_slide_df[(scandinavian_slide_df['Typo_Unk'] == True)].groupby('file').size()
scandinavian_slide_shib_distro.loc[:, 'sum'] = scandinavian_slide_shib_distro.sum(axis=1)
scandinavian_slide_shib_distro = pd.concat([scandinavian_slide_shib_distro, number_annots], axis=1)

scandinavian_slide_shib_distro['shibboleths: lexical'] = round(scandinavian_slide_shib_distro['lex shib'] \
                                                               / scandinavian_slide_shib_distro['total'] * 100, 1)
scandinavian_slide_shib_distro['shibboleths: other'] = round(scandinavian_slide_shib_distro['nonlex shib'] \
                                                               / scandinavian_slide_shib_distro['total'] * 100, 1)
scandinavian_slide_shib_distro['non shibboleths'] = round(scandinavian_slide_shib_distro['non shib'] \
                                                               / scandinavian_slide_shib_distro['total'] * 100, 1)
scandinavian_slide_shib_distro['other'] = round(scandinavian_slide_shib_distro['rest'] \
                                                               / scandinavian_slide_shib_distro['total'] * 100, 1)

scandinavian_slide_shib_distro.loc[:, 'perc_sum'] = scandinavian_slide_shib_distro[['shibboleths: lexical', 'shibboleths: other', 'non shibboleths', 'other']].sum(axis=1)

scandinavian_slide_shib_distro.loc['average'] = round(scandinavian_slide_shib_distro.mean(axis=0), 1)

scandinavian_slide_shib_distro

In [ ]:
scandinavian_slide_shib_distro = pd.concat([scandinavian_slide_shib_distro, scandinavian_slide_methods_df], axis=1)
scandinavian_slide_shib_distro

In [ ]:
scandinavian_slide_shib_distro = scandinavian_slide_shib_distro.sort_values(by=['method']).reset_index(names=['list']).set_index('method')
scandinavian_slide_shib_distro

In [ ]:
scandinavian_slide_overview = scandinavian_slide_shib_distro[['shibboleths: lexical', 'shibboleths: other', 'non shibboleths', 'other']]
scandinavian_slide_overview

## Generate graphs

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
#sns.set_theme(style="whitegrid")
#sns.set_color_codes("pastel")

In [ ]:
german_jodel_overview.plot(kind='bar', stacked=True, colormap="summer", title="Shibboleth distribution overview : german_jodel").figure.savefig('german_jodel_shib_overview.pdf', bbox_inches='tight')

In [ ]:
bcms_twitter_overview.plot(kind='bar', stacked=True, colormap="summer", title="Shibboleth distribution overview : bcms_twitter").figure.savefig('bcms_twitter_shib_overview.pdf', bbox_inches='tight')

In [ ]:
bcms_setimes_overview.plot(kind='bar', colormap="summer", stacked=True, title="Shibboleth distribution overview : bcms_setimes").figure.savefig('bcms_setimes_shib_overview.pdf', bbox_inches='tight')

In [ ]:
greek_dialects_overview.plot(kind='bar', stacked=True, colormap="summer", title="Shibboleth distribution overview : greek_dialects").figure.savefig('greek_dialects_shib_overview.pdf', bbox_inches='tight')

In [ ]:
finnish_suomi24_overview.plot(kind='bar', stacked=True, colormap="summer", title="Shibboleth distribution overview : finnish_suomi24").figure.savefig('finnish_suomi24_shib_overview.pdf', bbox_inches='tight')

In [ ]:
scandinavian_slide_overview.plot(kind='bar', stacked=True, colormap="summer", title="Shibboleth distribution overview : scandinavian_slide").figure.savefig('scandinavian_slide_shib_overview.pdf', bbox_inches='tight')

In [ ]:
estonian_voro_overview.plot(kind='bar', stacked=True, colormap="summer", title="Shibboleth distribution overview : estonian_voro").figure.savefig('estonian_voro_shib_overview.pdf', bbox_inches='tight')

## Generating csv tables with main results

In [ ]:
bcms_setimes_overview.to_csv('bcms_setimes_overview.csv')
bcms_twitter_overview.to_csv('bcms_twitter_overview.csv')
finnish_suomi24_overview.to_csv('finnish_suomi24_overview.csv')
german_jodel_overview.to_csv('german_jodel_overview.csv')
greek_dialects_overview.to_csv('greek_dialects_overview.csv')
scandinavian_slide_overview.to_csv('scandinavian_slide_overview.csv')
estonian_voro_overview.to_csv('estonian_voro_overview.csv')

# Analysis by method 

In [ ]:
bcms_setimes_overview.loc[:, 'dataset'] = 'bcms_setimes'
bcms_setimes_overview

In [ ]:
bcms_twitter_overview.loc[:, 'dataset'] = 'bcms_twitter'
bcms_twitter_overview

In [ ]:
finnish_suomi24_overview.loc[:, 'dataset'] = 'finnish_suomi24'
finnish_suomi24_overview

In [ ]:
german_jodel_overview.loc[:, 'dataset'] = 'german_jodel'
german_jodel_overview

In [ ]:
greek_dialects_overview.loc[:, 'dataset'] = 'greek_dialects'
greek_dialects_overview

In [ ]:
scandinavian_slide_overview.loc[:, 'dataset'] = 'scandinavian_slide'
scandinavian_slide_overview

In [ ]:
estonian_voro_overview.loc[:, 'dataset'] = 'estonian_voro'
estonian_voro_overview

In [ ]:
all_overview = pd.concat([bcms_twitter_overview, bcms_setimes_overview, finnish_suomi24_overview, \
                         german_jodel_overview, greek_dialects_overview, scandinavian_slide_overview, estonian_voro_overview], axis=0).reset_index(names=['method'])

all_overview

In [ ]:
all_overview[all_overview['method'] == 'ig_0.1'].set_index('dataset')

## Generate graphs by method

In [ ]:
all_overview[all_overview['method'] == 'ig_0.1'].set_index('dataset').plot(kind='bar', stacked=True, title="Shibboleth distribution overview : ig_0.1").figure.savefig('ig_0.1_shib_overview.pdf', bbox_inches='tight')

#all_overview[all_overview['method'] == 'ig_0.1'].set_index('dataset').plot(kind='bar', stacked=True, title="Shibboleth distribution overview : german_jodel").figure.savefig('german_jodel_shib_overview.pdf', bbox_inches='tight')

In [ ]:
all_overview[all_overview['method'] == 'ig_0.6'].set_index('dataset').plot(kind='bar', stacked=True, title="Shibboleth distribution overview : ig_0.6").figure.savefig('ig_0.6_shib_overview.pdf', bbox_inches='tight')

In [ ]:
all_overview[all_overview['method'] == 'lime_0.1'].set_index('dataset').plot(kind='bar', stacked=True, title="Shibboleth distribution overview : lime_0.1").figure.savefig('lime_0.1_shib_overview.pdf', bbox_inches='tight')

In [ ]:
all_overview[all_overview['method'] == 'lime_0.6'].set_index('dataset').plot(kind='bar', stacked=True, title="Shibboleth distribution overview : lime_0.6").figure.savefig('lime_0.6_shib_overview.pdf', bbox_inches='tight')

In [ ]:
all_overview[all_overview['method'] == 'shap_0.1'].set_index('dataset').plot(kind='bar', stacked=True, title="Shibboleth distribution overview : shap_0.1").figure.savefig('shap_0.1_shib_overview.pdf', bbox_inches='tight')

In [ ]:
all_overview[all_overview['method'] == 'shap_0.6'].set_index('dataset').plot(kind='bar', stacked=True, title="Shibboleth distribution overview : shap_0.6").figure.savefig('shap_0.6_shib_overview.pdf', bbox_inches='tight')

In [ ]:
all_overview[all_overview['method'] == 'loo_0.1'].set_index('dataset').plot(kind='bar', stacked=True, title="Shibboleth distribution overview : loo_0.1").figure.savefig('loo_0.1_shib_overview.pdf', bbox_inches='tight')

In [ ]:
all_overview[all_overview['method'] == 'loo_0.6'].set_index('dataset').plot(kind='bar', stacked=True, title="Shibboleth distribution overview : loo_0.6").figure.savefig('loo_0.6_shib_overview.pdf', bbox_inches='tight')

In [ ]:
import matplotlib.cm as cm
import numpy as np

In [ ]:
fig, axes = plt.subplots(nrows=1, ncols=3, sharey='row', figsize=(5,4))
all_overview[all_overview['dataset'] == 'scandinavian_slide'].set_index('method').plot(kind='bar', stacked=True, title="scandinavian_slide",  colormap="summer", ax=axes[0], legend=False)
all_overview[all_overview['dataset'] == 'bcms_setimes'].set_index('method').plot(kind='bar', stacked=True, title="bcms_setimes",  colormap="summer", ax=axes[1], legend=False)
all_overview[all_overview['dataset'] == 'estonian_voro'].set_index('method').plot(kind='bar', stacked=True, title="estonian_voro",  colormap="summer", ax=axes[2], legend=False)

axes[0].set_yticks(range(0, 105, 10))
axes[0].set_ylim(0, 105)

axes[0].set_xlabel(None)
axes[1].set_xlabel(None)

ax=axes[1]
handles, labels = ax.get_legend_handles_labels()
#fig.legend(handles, labels, loc='upper center')
plt.legend(handles, labels, loc='lower center', bbox_to_anchor=(1.2, 0.7), bbox_transform=plt.gcf().transFigure)
plt.subplots_adjust(left=0.1, right=0.9, 
                    top=0.9, bottom=0.1, 
                    wspace=0.3, hspace=0.5
                   )
fig.figure.savefig('experiment1-overview.pdf', bbox_inches='tight')


In [ ]:
fig, axes = plt.subplots(nrows=1, ncols=4, sharey='row', figsize=(10,4))
all_overview[all_overview['dataset'] == 'bcms_twitter'].set_index('method').plot(kind='bar', stacked=True, title="bcms_twitter",  colormap="summer", ax=axes[0], legend=False)
all_overview[all_overview['dataset'] == 'finnish_suomi24'].set_index('method').plot(kind='bar', stacked=True, title="finnish_suomi24",  colormap="summer", ax=axes[1], legend=False)
all_overview[all_overview['dataset'] == 'german_jodel'].set_index('method').plot(kind='bar', stacked=True, title="german_jodel",  colormap="summer", ax=axes[2], legend=False)
all_overview[all_overview['dataset'] == 'greek_dialects'].set_index('method').plot(kind='bar', stacked=True, title="greek_dialects",  colormap="summer", ax=axes[3], legend=False)                                                                                 
axes[0].set_yticks(range(0, 105, 10))
axes[0].set_ylim(0, 105)

axes[0].set_xlabel(None)
axes[1].set_xlabel(None)
axes[2].set_xlabel(None)
axes[3].set_xlabel(None)

ax=axes[2]
handles, labels = ax.get_legend_handles_labels()
#fig.legend(handles, labels, loc='upper center')
plt.legend(handles, labels, loc='lower center', bbox_to_anchor=(1.02, 0.7), bbox_transform=plt.gcf().transFigure)
plt.subplots_adjust(left=0.1, right=0.9, 
                    top=0.9, bottom=0.1, 
                    wspace=0.3, hspace=0.5
                   )
fig.figure.savefig('experiment2-overview-per_dataset.pdf', bbox_inches='tight')

In [ ]:
fig, axes = plt.subplots(nrows=2, ncols=4, sharey='row', figsize=(8,11))
subset = ['bcms_twitter', 'finnish_suomi24', 'german_jodel', 'greek_dialects']
all_overview[(all_overview['method'] == 'ig_0.1') & (all_overview['dataset'].isin(subset))].set_index('dataset').plot(kind='bar', stacked=True, title="ig_0.1",  colormap="summer", ax=axes[0,0], legend=False)
all_overview[(all_overview['method'] == 'lime_0.1') & (all_overview['dataset'].isin(subset))].set_index('dataset').plot(kind='bar', stacked=True, title="lime_0.1",  colormap="summer", ax=axes[0,1], legend=False)
all_overview[(all_overview['method'] == 'loo_0.1') & (all_overview['dataset'].isin(subset))].set_index('dataset').plot(kind='bar', stacked=True, title="loo_0.1",  colormap="summer", ax=axes[0,2], legend=False)
all_overview[(all_overview['method'] == 'shap_0.1') & (all_overview['dataset'].isin(subset))].set_index('dataset').plot(kind='bar', stacked=True, title="shap_0.1",  colormap="summer", ax=axes[0,3], legend=False)                                                                                 
all_overview[(all_overview['method'] == 'ig_0.6') & (all_overview['dataset'].isin(subset))].set_index('dataset').plot(kind='bar', stacked=True, title="ig_0.6",  colormap="summer", ax=axes[1,0], legend=False)
all_overview[(all_overview['method'] == 'lime_0.6') & (all_overview['dataset'].isin(subset))].set_index('dataset').plot(kind='bar', stacked=True, title="lime_0.6",  colormap="summer", ax=axes[1,1], legend=False)
all_overview[(all_overview['method'] == 'loo_0.6') & (all_overview['dataset'].isin(subset))].set_index('dataset').plot(kind='bar', stacked=True, title="loo_0.6",  colormap="summer", ax=axes[1,2], legend=False)
all_overview[(all_overview['method'] == 'shap_0.6') & (all_overview['dataset'].isin(subset))].set_index('dataset').plot(kind='bar', stacked=True, title="shap_0.6",  colormap="summer", ax=axes[1,3], legend=False) 

axes[0,1].set_yticks(range(0, 105, 10))
axes[0,1].set_ylim(0, 105)

axes[1,0].set_yticks(range(0, 105, 10))
axes[1,0].set_ylim(0, 105)

axes[0,0].set_xlabel(None)
axes[0,1].set_xlabel(None)
axes[0,2].set_xlabel(None)
axes[0,3].set_xlabel(None)
axes[1,0].set_xlabel(None)
axes[1,1].set_xlabel(None)
axes[1,2].set_xlabel(None)
axes[1,3].set_xlabel(None)

ax=axes[1,3]
handles, labels = ax.get_legend_handles_labels()
#fig.legend(handles, labels, loc='upper center')
plt.legend(handles, labels, loc='lower center', bbox_to_anchor=(1.02, 0.7), bbox_transform=plt.gcf().transFigure)
plt.subplots_adjust(left=0.1, right=0.9, 
                    top=0.9, bottom=0.1, 
                    wspace=0.3, hspace=0.5
                   )
fig.figure.savefig('experiment2-overview-per_method.pdf', bbox_inches='tight')

In [ ]:
all_overview[all_overview['method'].isin(['ig_0.1', 'ig_0.6'])]

In [ ]:
fig, axes = plt.subplots(nrows=2, ncols=4, sharey='row', figsize=(10,11))

all_overview[all_overview['method'] == 'loo_0.1'].set_index('dataset').plot(kind='bar', stacked=True, title="loo_0.1",  colormap="summer", ax=axes[0,0], legend=False)
all_overview[all_overview['method'] == 'loo_0.6'].set_index('dataset').plot(kind='bar', stacked=True, title="loo_0.6",  colormap="summer", ax=axes[1,0], legend=False)
all_overview[all_overview['method'] == 'lime_0.1'].set_index('dataset').plot(kind='bar', stacked=True, title="lime_0.1",  colormap="summer", ax=axes[0,1], legend=False)
all_overview[all_overview['method'] == 'lime_0.6'].set_index('dataset').plot(kind='bar', stacked=True, title="lime_0.6",  colormap="summer", ax=axes[1,1], legend=False)
all_overview[all_overview['method'] == 'shap_0.1'].set_index('dataset').plot(kind='bar', stacked=True, title="shap_0.1",  colormap="summer", ax=axes[0,2], legend=False)
all_overview[all_overview['method'] == 'shap_0.6'].set_index('dataset').plot(kind='bar', stacked=True, title="shap_0.6",  colormap="summer", ax=axes[1,2], legend=False)
all_overview[all_overview['method'] == 'ig_0.1'].set_index('dataset').plot(kind='bar', stacked=True, title="ig_0.1",  colormap="summer", ax=axes[0,3], legend=False)
all_overview[all_overview['method'] == 'ig_0.6'].set_index('dataset').plot(kind='bar', stacked=True, title="ig_0.6",  colormap="summer", ax=axes[1,3], legend=False)

axes[0,0].set_yticks(range(0, 105, 10))
axes[0,0].set_ylim(0, 105)


axes[1,0].set_yticks(range(0, 105, 10))
axes[1,0].set_ylim(0, 105)

axes[0,0].set_xlabel(None)
axes[0,1].set_xlabel(None)
axes[0,2].set_xlabel(None)
axes[0,3].set_xlabel(None)
axes[1,0].set_xlabel(None)
axes[1,1].set_xlabel(None)
axes[1,2].set_xlabel(None)
axes[1,3].set_xlabel(None)

ax=axes[1,3]
handles, labels = ax.get_legend_handles_labels()
#fig.legend(handles, labels, loc='upper center')
plt.legend(handles, labels, loc='lower center', bbox_to_anchor=(1, 0.8), bbox_transform=plt.gcf().transFigure)
plt.subplots_adjust(left=0.1, right=0.9, 
                    top=0.9, bottom=0.1, 
                    wspace=0.3, hspace=0.5)

In [ ]:

all_overview

In [ ]:
df_list = [bcms_twitter_overview, bcms_setimes_overview]

In [ ]:
def plot_clustered_stacked(dfall, labels=None, title="multiple stacked bar plot",  H="/", **kwargs):
    """Given a list of dataframes, with identical columns and index, create a clustered stacked bar plot. 
labels is a list of the names of the dataframe, used for the legend
title is a string for the title of the plot
H is the hatch used for identification of the different dataframe"""

    n_df = len(dfall)
    n_col = len(dfall[0].columns) 
    n_ind = len(dfall[0].index)
    axe = plt.subplot(111)

    for df in dfall : # for each data frame
        axe = df.plot(kind="bar",
                      linewidth=0,
                      stacked=True,
                      ax=axe,
                      legend=False,
                      grid=False,
                      **kwargs)  # make bar plots

    h,l = axe.get_legend_handles_labels() # get the handles we want to modify
    for i in range(0, n_df * n_col, n_col): # len(h) = n_col * n_df
        for j, pa in enumerate(h[i:i+n_col]):
            for rect in pa.patches: # for each index
                rect.set_x(rect.get_x() + 1 / float(n_df + 1) * i / float(n_col))
                rect.set_hatch(H * int(i / n_col)) #edited part     
                rect.set_width(1 / float(n_df + 1))

    axe.set_xticks((np.arange(0, 2 * n_ind, 2) + 1 / float(n_df + 1)) / 2.)
    axe.set_xticklabels(df.index, rotation = 0)
    axe.set_title(title)

    # Add invisible data to add another legend
    n=[]        
    for i in range(n_df):
        n.append(axe.bar(0, 0, color="gray", hatch=H * i))

    l1 = axe.legend(h[:n_col], l[:n_col], loc=[1.01, 0.5])
    if labels is not None:
        l2 = plt.legend(n, labels, loc=[1.01, 0.1]) 
    axe.add_artist(l1)
    return axe


In [ ]:
plot_clustered_stacked(df_list, ['bcms_twitter', 'bcms_setimes'])

In [ ]:
all_overview

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(nrows=4, ncols=2, figsize=(12, 15))
fig.suptitle('Category distribution among shibboleths')
axe = axes.ravel()

for i in range(1,9):
    file_id = 'list' + str(i)
    full_df[(full_df['file'] == file_id) & (full_df['shibboleth'] == 'yes')].groupby('feature').size().plot.pie(ax=axe[i-1], title = file_id, autopct='%1.1f%%')
    
fig.savefig('category_distro_shibboleths-twitter.pdf', dpi=300)

In [ ]:
fig, axes = plt.subplots(nrows=4, ncols=2, figsize=(12, 15))
fig.suptitle('Non-shibboleths, shibboleths, non minimal shibboleths')
axe = axes.ravel()

for i in range(1,9):
    file_id = 'list' + str(i)
    full_df[(full_df['file'] == file_id)].groupby('shibboleth').size().plot.pie(ax=axe[i-1], title = file_id, autopct='%1.1f%%')
    
fig.savefig('shibboleths-nonminimal-setimes.pdf', dpi=300)